In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
import yaml

In [ ]:
with open("settings.yaml", "r") as file:
    settings = yaml.safe_load(file)

cmc_file = settings["cmc"]["file"]
cmc_sheet = settings["cmc"]["sheet"]


In [ ]:
df = pd.read_excel(cmc_file, sheet_name=0)

In [ ]:
df

In [ ]:
a = df.columns.tolist()
a.sort()
a
df.columns

In [ ]:
unused_cols = ["closed_time", "note", "appliance_site", "additional_description", "assertion_source", "alert_info", "playbook_contents", "custom_fields_src", "custom_fields_dst", "physical_links"]
used_cols = [col for col in df.columns if col not in unused_cols]
len(used_cols), len(unused_cols)

In [ ]:
df[used_cols]
df_reduced = df[used_cols]

In [ ]:
df_reduced

In [ ]:
df_reduced[["name"]].value_counts(sort=True)

In [ ]:
my_dict = {}
for c in used_cols:
    c_dict = {}
    c_dict["n_unique"] = df_reduced[c].nunique()
    c_dict["top_3"] = [str(x) for x in df_reduced[c].value_counts(sort=True).head(3).index.tolist()]
    my_dict[c] = c_dict

In [ ]:
# save as json
# import json
# with open("my_dict.json", "w") as f:
#     json.dump(my_dict, f)

In [ ]:
# group by per name, take the count, and mean-min-max of risk
# use style to highlighy is_security and is_incident


out = (
    df_reduced
    .groupby("name")
    .agg({
        "name": "count",
        "protocol": "unique",
        "risk": ["min", "max"],
        "is_security": "mean",
        "is_incident": "mean",
        # "counter": ["min", "max"],
    })
)

# Arrotonda solo le colonne numeriche interessate
# Attenzione: le colonne hanno MultiIndex, es. ("risk","min"), ("is_security","mean"), ecc.
out[("protocol", "unique")] = out[("protocol", "unique")].apply(lambda x: ", ".join(map(str, x)))
out[("risk", "min")] = out[("risk", "min")].astype(int)
out[("risk", "max")] = out[("risk", "max")].astype(int)
out[("is_security", "mean")] = out[("is_security", "mean")].astype(int)
out[("is_incident", "mean")] = out[("is_incident", "mean")].astype(int)
# Se counter è intero, puoi evitarlo; altrimenti:
# out[("counter", "min")] = out[("counter", "min")].round(0)
# out[("counter", "max")] = out[("counter", "max")].round(0)

# Ordina usando la colonna di count (MultiIndex: ("name","count"))
out = out.sort_values(("name", "count"), ascending=False)
out[("is_security", "mean")] = out[("is_security", "mean")].astype(bool)
out[("is_incident", "mean")] = out[("is_incident", "mean")].astype(bool)

out
(out
.style
.background_gradient(subset=[("name", "count")], cmap="Blues")

.map(
    lambda v: "background-color: red" if v == 1 else "background-color: blue",
    subset=[("is_security", "mean"), ("is_incident", "mean")],
)
.background_gradient(subset=["risk"], cmap="Reds")
)

In [ ]:
df_reduced[["name", "description"]].value_counts(sort=True).head(20)

In [ ]:
df_reduced['protocol'].unique()

In [ ]:
print(df_reduced.describe())

In [ ]:
df_time = df_reduced[["time"]]
df_time["time"] = pd.to_datetime(df_time["time"])
df_time["day"] = df_time["time"].dt.date
df_time["hour"] = df_time["time"].dt.hour

In [ ]:
df_time

In [ ]:
df_event_per_hour = df_time.groupby("hour").size()
plt.figure(figsize=(10, 4))
df_event_per_hour.plot(kind="bar")
plt.title("Events per Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Number of Events")

In [ ]:
df_event_per_date = df_time.groupby("day").size()
# seaborn line plot

plt.figure(figsize=(10, 4))
sns.lineplot(x=df_event_per_date.index, y=df_event_per_date.values)
# grid and title
plt.grid(True)
plt.title("Events per Day")
plt.ylabel("Number of Events")
plt.xticks(rotation=45)
# ticks every 7 days
plt.xticks(df_event_per_date.index[::7])
# global font size
plt.rcParams.update({"font.size": 18})